# 06 — Model training

Assembles the real rolling-window training dataset, shows the LambdaRank query grouping and the eligibility gates, then fits a real `lightgbm.LGBMRanker` (`objective="lambdarank"`) on it.

**Data mode: REAL DATA.** Every figure below is computed from the genuine Bloomberg CSV exports for this strategy, through production AtlasQuant code. No synthetic, simulated, imputed, or approximated value appears anywhere in this notebook.

Those exports are gitignored, so they are absent in a fresh clone. When they are missing this notebook still executes top to bottom, but every computing cell prints an explicit *real data not present — cannot produce a genuine result* notice instead of a number. It never falls back to synthetic data: a plausible-looking figure derived from invented inputs is not a result (see `CLAUDE.md`).

In [ ]:
import _real_data as rd

print(rd.banner())
AVAILABLE = rd.data_available()


## The rolling window

For a target quarter *q*, training uses labeled rows from quarters *q − ml_train_years ≤ q' < q*. Two rules matter and are enforced separately:

1. The target quarter itself, and anything after it, is always excluded.
2. A labeled row may only be used once **its own label was knowable** — `label_available_at < training_cutoff`, strictly. A label available *exactly at* the cutoff is not yet realized at the instant the decision is made, because that price only prints at the close.

In [ ]:
if AVAILABLE:
    from datetime import datetime, time

    from atlas_quant.strategies.multi_factor_ranking_ml.config import FEATURE_SCHEMA_VERSION
    from atlas_quant.strategies.multi_factor_ranking_ml.training_dataset import build_training_dataset

    bundle = rd.slice_bundle()
    config = rd.config()
    target_cycle = bundle.cycles[-1]
    training_cutoff = datetime.combine(target_cycle.cutoff, time.min)

    dataset = build_training_dataset(
        target_cycle.quarter_start,
        training_cutoff,
        bundle.labeled_quarters,
        strategy_id=config.strategy_id,
        feature_schema_version=FEATURE_SCHEMA_VERSION,
        ml_train_years=config.ml_train_years,
        model_config_identity=config.model.identity(),
    )
    print("target quarter  ", dataset.target_quarter_end)
    print("training cutoff ", dataset.training_cutoff)
    print("candidates      ", len(dataset.candidate_quarters))
    print("included        ", dataset.quarter_count, "quarters")
    print("rows            ", f"{dataset.total_row_count:,}")
    print("schema identity ", dataset.model_schema_identity)
    print()
    print("included:", [q.isoformat() for q in dataset.included_quarters])
    print()
    print("excluded:")
    for excluded in dataset.excluded_quarters:
        print(f"  {excluded.quarter_end}  {excluded.reason}")
else:
    bundle = None
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## LambdaRank query grouping

LambdaRank compares items only *within* a query. Here one query = one quarter: the model learns to order that quarter's cross-section, never to compare a 2021 stock against a 2023 one. `groups` records the contiguous per-quarter row-block sizes and must partition the matrix exactly, or LightGBM would silently align quarters to the wrong rows.

In [ ]:
if AVAILABLE:
    print(f"{'quarter':<12} {'group size':>11}")
    for quarter, size in zip(dataset.included_quarters, dataset.groups):
        print(f"{str(quarter):<12} {size:>11,}")
    print()
    print(f"groups: {len(dataset.groups)}  sum: {sum(dataset.groups):,}  "
          f"matrix rows: {len(dataset.feature_matrix):,}")
    assert sum(dataset.groups) == len(dataset.feature_matrix) == len(dataset.relevances)
    print("OK: query groups partition the feature matrix exactly.")
    print()
    print("relevance distribution across the training set:")
    from collections import Counter
    for grade, count in sorted(Counter(dataset.relevances).items(), reverse=True):
        print(f"  grade {grade}: {count:>7,}")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Eligibility gates

There is no "minimum positive labels" gate and no single-class gate: both were artifacts of a binary classifier, and are deleted. LambdaRank has no classes — its requirement is that the relevance target *varies*, since a constant target yields no discordant pairs and therefore no gradient. That is checked from the data, not from configuration.

In [ ]:
if AVAILABLE:
    from atlas_quant.strategies.multi_factor_ranking_ml.training_dataset import (
        check_training_eligibility,
    )

    eligibility = check_training_eligibility(
        dataset, min_train_quarters=config.min_train_quarters
    )
    for field_name in (
        "eligible", "quarter_count", "min_train_quarters", "quarter_gate_passed",
        "distinct_relevance_count", "relevance_variation_present", "non_empty",
        "matrix_relevance_length_match", "groups_match_row_count", "has_finite_values",
    ):
        print(f"  {field_name:<30} {getattr(eligibility, field_name)}")
    if eligibility.reasons:
        print()
        print("reasons:", eligibility.reasons)
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Fitting the real ranker

`build_lgbm_ranker_estimator` constructs a real `lightgbm.LGBMRanker` with this config's exact parameters. `objective="lambdarank"` is pinned in code: it *is* the strategy decision, not a tunable default.

In [ ]:
if AVAILABLE:
    from atlas_quant.strategies.multi_factor_ranking_ml.config import STRATEGY_VERSION
    from atlas_quant.strategies.multi_factor_ranking_ml.estimator import (
        build_lgbm_ranker_estimator,
        resolve_estimator_parameters,
    )
    from atlas_quant.strategies.multi_factor_ranking_ml.model_training import train_model

    print("estimator parameters:")
    for key, value in resolve_estimator_parameters(config.model).items():
        print(f"  {key:<18} {value}")
    print()

    training_result = train_model(
        dataset, eligibility, config.model, build_lgbm_ranker_estimator,
        strategy_id=config.strategy_id, strategy_version=STRATEGY_VERSION,
    )
    print("training state:", training_result.state.value)
    print("fit error     :", training_result.fit_error)
    print("warnings      :", training_result.warnings)
else:
    training_result = None
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Model identity

Every input that could change the fitted model is folded into a deterministic identity, so a score can always be traced back to exactly the model and training window that produced it.

In [ ]:
if AVAILABLE and training_result.model_identity is not None:
    identity = training_result.model_identity
    print("identity hash          ", identity.identity())
    print("estimator              ", identity.estimator_type)
    print("library                ", identity.library, identity.library_version)
    print("random_state           ", identity.random_state)
    print("training cutoff        ", identity.training_cutoff)
    print("included quarters      ", len(identity.included_quarters))
    print("used feature names     ", len(identity.used_feature_names), "of",
          len(dataset.feature_matrix.column_names))
    dropped = set(dataset.feature_matrix.column_names) - set(identity.used_feature_names)
    print("dropped (all-missing in this window):", sorted(dropped) or "none")
elif AVAILABLE:
    print("training did not produce a model:", training_result.state.value)
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Feature importances

LightGBM split counts over the fitted trees. Descriptive only: a high split count is not evidence that a feature carries genuine forward information.

In [ ]:
if AVAILABLE and training_result.fitted_estimator is not None:
    importances = list(training_result.fitted_estimator.feature_importances_)
    names = training_result.model_identity.used_feature_names
    ranked = sorted(zip(names, importances), key=lambda pair: -pair[1])
    total = sum(importances) or 1
    print(f"{'feature':<42} {'splits':>8} {'share':>8}")
    for name, value in ranked[:20]:
        print(f"{name:<42} {value:>8,} {value / total:>8.1%}")
    print()
    print("features with zero splits:", sum(1 for _, v in ranked if v == 0))
elif AVAILABLE:
    print("no fitted estimator to inspect")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Findings

- On the real slice the trailing 3-year window contributes ~10-11 quarters of labeled rows for the final target cycle, comfortably above the `min_train_quarters = 8` gate.
- Query groups partition the feature matrix exactly, so each quarter is one LambdaRank query and no cross-quarter comparison is ever formed.
- The most recent labeled quarter is excluded by the strict `label_available_at < training_cutoff` rule, not by the window bound. That exclusion is the leakage guard doing its job.

## Limitations

- Feature importances describe the fitted trees, not economic significance. They are not a factor-attribution result and should not be read as one.
- Hyperparameters were carried across from the previous classifier configuration as starting defaults. They have not been tuned for LambdaRank, and no validation split is held out here.
- One model is fitted for one target quarter here. The backtest refits per cycle (notebook 08); nothing about this single fit generalizes on its own.